# Crisis Negotiator — GRPO Training (Colab-ready)

Trains **Qwen2.5-3B-Instruct** with GRPO on the Crisis Negotiator RL environment.

| | |
|---|---|
| **Hardware** | Colab T4 (15 GB) or local RTX 4090 |
| **Runtime** | ~35–50 min for 64 prompts × 8 generations |
| **Output** | `crisis-negotiator-trained/` LoRA adapter + `crisis_training_log.json` + `reward_curve_training.png` |

**What this trains:** An LLM that acts as an FBI-style crisis negotiator. It must simultaneously
de-escalate a hostage-taker, resist tactical pressure from a commander, and use BCSM techniques —
all optimised by sparse environment rewards. No imitation data. No scripted dialogue.

**Reward signal:** Blended `0.6×heuristic + 0.4×env_grader + outcome_bonus`.
Surrender/hostage-release earns +0.40; harm/tactical-intervention earns −0.30.

In [ ]:
# Cell 1 — Install dependencies
# openenv-core : the RL environment framework (CrisisNegotiatorEnvironment)
# trl 0.14+    : GRPOTrainer
# peft         : LoRA adapter
# sentence-transformers : emotion reward (RLVER-style cosine similarity)
!pip install -q "openenv-core>=0.2.3" "trl>=0.14" peft accelerate datasets sentence-transformers matplotlib
# torch + CUDA are pre-installed on Colab. On a local GPU see requirements-train.txt
print('Dependencies installed.')

In [ ]:
# Cell 2 — Clone repo (skip if running inside the repo folder already)
import os, pathlib

if not pathlib.Path('server').exists():
    # Replace with your published repo URL
    !git clone https://github.com/Dinesh052/crisis-negotiator-openenv.git .
    print('Cloned.')
else:
    print('Already inside repo.')

print('Files:', sorted(f for f in os.listdir('.') if not f.startswith('.'))[:14])

In [ ]:
# Cell 3 — Smoke-test the environment before training
# Verifies: env resets, step() works, reward is a float, grader.py is reachable.
import sys
sys.path.insert(0, '.')

from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
print(f'Reset OK  | phase={obs.phase}  demands={[d["text"] for d in obs.stated_demands]}')

step_obs = env.step(NegotiatorAction(
    action_type='emotional_label',
    content="It sounds like you're feeling completely overwhelmed right now.",
    reasoning='open with empathy',
    target='hostage_taker',
))
print(f'Step OK   | reward={step_obs.reward:.4f}  phase={step_obs.phase}  done={step_obs.done}')

assert isinstance(step_obs.reward, float), 'reward must be a float!'
print('\nEnvironment smoke-test PASSED ✓')

In [ ]:
# Cell 4 — Run GRPO training
# train_local.py handles everything: model load (3B bf16 LoRA), dataset build,
# GRPOTrainer loop, crisis_training_log.json logging, adapter save.
#
# On Colab T4 (15 GB VRAM), 3B bf16 fits with gradient-checkpointing.
# If you hit OOM, uncomment the next line to use the 1.5B model instead:
# !sed -i 's/Qwen2.5-3B-Instruct/Qwen2.5-1.5B-Instruct/' train_local.py

import os, subprocess, sys
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'

# Run training — output streams directly to this cell
result = subprocess.run(
    [sys.executable, 'train_local.py'],
    capture_output=False,
    text=True,
)
print(f'\nTraining finished. Exit code: {result.returncode}')

In [ ]:
# Cell 5 — Plot training reward curve
# Reads crisis_training_log.json (written by train_local.py every 2 steps).
# Shows per-step reward coloured by curriculum phase + rolling mean.
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open('crisis_training_log.json') as f:
    log = json.load(f)

steps   = [e['global_step'] for e in log]
rewards = [e['reward']      for e in log]
phases  = [e.get('obs_phase', 'opening') for e in log]

# Rolling mean
window  = min(16, len(rewards))
rolling = np.convolve(rewards, np.ones(window) / window, mode='valid')
roll_x  = steps[window - 1:]

# Phase colours
PHASE_COLORS = {'opening': '#4C72B0', 'negotiation': '#DD8452', 'resolution': '#55A868'}
colours = [PHASE_COLORS.get(p, '#999999') for p in phases]

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(steps, rewards, c=colours, alpha=0.35, s=14, zorder=2)
ax.plot(roll_x, rolling, color='black', linewidth=2, label=f'rolling mean (w={window})', zorder=3)
ax.axhline(0.755, color='grey',    linestyle='--', linewidth=1, label='random baseline 0.755')
ax.axhline(0.950, color='#2ca02c', linestyle='--', linewidth=1, label='heuristic baseline 0.950')
ax.set_xlabel('GRPO Training Step')
ax.set_ylabel('Blended Episode Reward')
ax.set_title('Crisis Negotiator — GRPO Training Progress')

# Combined legend: lines + phase patches
phase_patches = [mpatches.Patch(color=c, label=p) for p, c in PHASE_COLORS.items()]
line_handles, line_labels = ax.get_legend_handles_labels()
ax.legend(handles=line_handles + phase_patches,
          labels=line_labels + list(PHASE_COLORS.keys()),
          fontsize=7, loc='lower right', ncol=2)

plt.tight_layout()
plt.savefig('reward_curve_training.png', dpi=150)
plt.show()

print(f'Steps: {len(log)}')
print(f'Mean reward (all)  : {np.mean(rewards):.4f}')
print(f'Mean reward (last 10): {np.mean(rewards[-10:]):.4f}')
print(f'Improvement vs first 10: {np.mean(rewards[-10:]) - np.mean(rewards[:10]):+.4f}')
print('Saved reward_curve_training.png')

In [ ]:
# Cell 6 — Eval: compare random / heuristic / trained (n=10 per difficulty)
import subprocess, sys, json, pathlib

result = subprocess.run(
    [
        sys.executable, 'eval_baselines.py',
        '--n', '10',
        '--difficulties', 'easy,medium,hard',
        '--include-trained',
        '--adapter-dir', './crisis-negotiator-trained',
        '--base-model', 'Qwen/Qwen2.5-3B-Instruct',
    ],
    capture_output=False,
    text=True,
)
print('Exit code:', result.returncode)

if pathlib.Path('eval_summary.json').exists():
    summary = json.loads(pathlib.Path('eval_summary.json').read_text())
    print('\n=== EVALUATION RESULTS ===')
    print(f'{"Policy":15s}  {"Reward":>8s}  {"Surrender":>10s}  {"Steps":>6s}')
    print('-' * 46)
    for policy, s in summary.items():
        print(f"{policy:15s}  {s['mean_final_reward']:8.3f}  "
              f"{s['surrender_rate']:10.0%}  {s['mean_steps']:6.1f}")

In [ ]:
# Cell 7 — Download artifacts (Colab only)
try:
    from google.colab import files
    files.download('reward_curve_training.png')
    files.download('crisis_training_log.json')
    files.download('eval_summary.json')
    print('Downloads triggered.')
except ImportError:
    print('Not running in Colab — files are in the current directory.')